# 🎙️ Emotion Recognition — WavLM-Large + 4 Datasets
### Optimized for Kaggle (P100/T4 GPU)

**Setup:**
- Model: `microsoft/wavlm-large` (317M params)
- Datasets: RAVDESS + CREMA-D + TESS + SAVEE (~12,162 samples)
- Platform: Kaggle (12hr session, 16GB VRAM)

**To enable GPU on Kaggle:** Settings → Accelerator → GPU P100 (recommended) or T4

## Step 1: Install Dependencies

In [2]:
!pip install transformers==4.47.1 torch==2.7.1+cu118 torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118 -q


In [3]:
import torch
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
print(torch.__version__)   # 2.7.1+cu118
print("✅ imports OK")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

2.7.1+cu118
✅ imports OK


## Step 2: Check GPU Availability

In [37]:
import torch

if torch.cuda.is_available():
    gpu_name         = torch.cuda.get_device_name(0)
    total_memory     = torch.cuda.get_device_properties(0).total_memory / 1e9
    reserved_memory  = torch.cuda.memory_reserved(0) / 1e9
    allocated_memory = torch.cuda.memory_allocated(0) / 1e9
    free_memory      = total_memory - reserved_memory

    print(f'GPU: {gpu_name}')
    print('─' * 40)
    print(f'Total VRAM:     {total_memory:.2f} GB')
    print(f'Used VRAM:      {allocated_memory:.2f} GB')
    print(f'Reserved VRAM:  {reserved_memory:.2f} GB')
    print(f'Free VRAM:      {free_memory:.2f} GB')
    print('─' * 40)

    usage_pct = (reserved_memory / total_memory) * 100
    bar = '█' * int(usage_pct / 2) + '░' * (50 - int(usage_pct / 2))
    print(f'Usage: [{bar}] {usage_pct:.1f}%')

    if free_memory < 2:
        print('\n🔴 WARNING — Almost out of VRAM!')
    elif free_memory < 4:
        print('\n🟡 CAUTION — VRAM getting low.')
    else:
        print('\n🟢 VRAM is fine.')
else:
    print('❌ No GPU — go to Settings → Accelerator → GPU P100')

GPU: Tesla P100-PCIE-16GB
────────────────────────────────────────
Total VRAM:     17.06 GB
Used VRAM:      2.53 GB
Reserved VRAM:  2.54 GB
Free VRAM:      14.52 GB
────────────────────────────────────────
Usage: [███████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 14.9%

🟢 VRAM is fine.


## Step 3: Import Libraries

In [38]:
!pip install evaluate --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


In [4]:
import os
import io
import gc
import json
import shutil
import numpy as np
import torch
import librosa
import soundfile as sf
from collections import Counter
from datasets import load_dataset, Audio, concatenate_datasets, Value
from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import evaluate
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from IPython.display import Audio as AudioPlayer, display

# ── Global constants — used consistently everywhere ─────────────────
SAMPLING_RATE    = 16000
MAX_AUDIO_LENGTH = 16000 * 5   # 5 seconds
device           = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('✅ All libraries imported!')
print(f'   Device: {device}')

2026-03-25 19:23:20.016047: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774466600.222812     280 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774466600.284863     280 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774466600.802480     280 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774466600.802526     280 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774466600.802530     280 computation_placer.cc:177] computation placer alr

✅ All libraries imported!
   Device: cuda


## Step 4: Load & Merge All 4 Datasets

| Dataset | Samples | Emotions |
|---|---|---|
| RAVDESS | ~1,440 | All 8 |
| CREMA-D | ~7,442 | 6 (no calm/surprised) |
| TESS | ~2,800 | 7 (no calm) |
| SAVEE | ~480 | 7 (no calm) |
| **Total** | **~12,162** | **All 8** |

In [5]:
# ── Emotion definitions ─────────────────────────────────────────────
emotion_labels = {
    0: 'neutral', 1: 'calm',    2: 'happy', 3: 'sad',
    4: 'angry',   5: 'fearful', 6: 'disgust', 7: 'surprised'
}
id2label = emotion_labels
label2id = {v: k for k, v in id2label.items()}

# ── Load RAVDESS ────────────────────────────────────────────────────
print('Loading RAVDESS...')
ravdess = load_dataset('xbgoose/ravdess')
ravdess = ravdess.cast_column('audio', Audio(decode=False))

def label_ravdess(example):
    filename     = example['audio']['path'].split('/')[-1]
    emotion_code = int(filename.split('-')[2])
    example['label'] = emotion_code - 1
    return example

ravdess = ravdess['train'].map(label_ravdess)
ravdess = ravdess.cast_column('label', Value('int64'))
print(f'   RAVDESS: {len(ravdess)} samples (all 8 emotions)')

# ── Load CREMA-D ────────────────────────────────────────────────────
print('Loading CREMA-D...')
cremad = load_dataset('mteb/crema-d', split='train')
cremad = cremad.cast_column('audio', Audio(decode=False))
cremad = cremad.cast_column('label', Value('int64'))

# mteb/crema-d: 0=Anger, 1=Disgust, 2=Fear, 3=Happy, 4=Neutral, 5=Sad
CREMAD_TO_SHARED = {
    0: label2id['angry'],   1: label2id['disgust'],
    2: label2id['fearful'], 3: label2id['happy'],
    4: label2id['neutral'], 5: label2id['sad'],
}

def label_cremad(example):
    example['label'] = CREMAD_TO_SHARED[example['label']]
    return example

cremad = cremad.map(label_cremad)
print(f'   CREMA-D: {len(cremad)} samples (6 emotions — no calm/surprised)')




    
# ── Align columns & merge all 4 ─────────────────────────────────────
keep_cols = ['audio', 'label']
ravdess   = ravdess.select_columns(keep_cols)
cremad    = cremad.select_columns(keep_cols)


combined = concatenate_datasets([ravdess, cremad]).shuffle(seed=42)
print(f'\n✅ Combined total: {len(combined)} samples')

# ── Class distribution ──────────────────────────────────────────────
counts = Counter(combined['label'])
print('\nClass distribution:')
for label_id, count in sorted(counts.items()):
    bar  = '█' * (count // 50)
    note = ' ← RAVDESS+SAVEE only' if emotion_labels[label_id] == 'calm' else ''
    print(f'  {emotion_labels[label_id]:12s} {bar} {count}{note}')

# ── Train / validation split ────────────────────────────────────────
train_test = combined.train_test_split(test_size=0.2, seed=42)
train_ds   = train_test['train']
eval_ds    = train_test['test']

print(f'\n   Training samples:   {len(train_ds)}')
print(f'   Validation samples: {len(eval_ds)}')

Loading RAVDESS...
   RAVDESS: 1440 samples (all 8 emotions)
Loading CREMA-D...
   CREMA-D: 7442 samples (6 emotions — no calm/surprised)
Loading TESS...
   TESS: 2800 samples (7 emotions — no calm)
Loading SAVEE...
   SAVEE: 480 samples (7 emotions — no calm)

✅ Combined total: 12162 samples

Class distribution:
  neutral      ██████████████████████████████████ 1703
  calm         ███ 192 ← RAVDESS+SAVEE only
  happy        ██████████████████████████████████████ 1923
  sad          ██████████████████████████████████████ 1923
  angry        ██████████████████████████████████████ 1923
  fearful      ██████████████████████████████████████ 1923
  disgust      ██████████████████████████████████████ 1923
  surprised    █████████████ 652

   Training samples:   9729
   Validation samples: 2433


## Step 5: Load WavLM-Large

Upgraded from `wavlm-base-plus` (94M) to `wavlm-large` (317M).
WavLM-large was trained on 94,000 hours of speech — significantly stronger representations.

In [6]:
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

model_name = 'microsoft/wavlm-large'
print(f'Loading pre-trained model: {model_name}')

feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)

model = AutoModelForAudioClassification.from_pretrained(
    model_name,
    num_labels=len(emotion_labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# Freeze feature encoder — protects pretrained weights early on
model.freeze_feature_encoder()
model = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n✅ Model loaded!')
print(f'   Total parameters:     {total_params:,}')
print(f'   Trainable parameters: {trainable_params:,}')
print(f'   Device: {device}')

Loading pre-trained model: microsoft/wavlm-large


Some weights of WavLMForSequenceClassification were not initialized from the model checkpoint at microsoft/wavlm-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



✅ Model loaded!
   Total parameters:     315,717,576
   Trainable parameters: 311,510,984
   Device: cuda


## Step 6: Preprocess Audio Data

In [16]:
gc.collect()

def preprocess_function(examples):
    audio_arrays = []
    for audio_dict in examples['audio']:
        audio_bytes  = audio_dict['bytes']
        audio_array, sr = sf.read(io.BytesIO(audio_bytes))
        audio_array  = np.array(audio_array, dtype=np.float32).flatten()
        if sr != SAMPLING_RATE:
            audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=SAMPLING_RATE)
        audio_arrays.append(audio_array)
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=SAMPLING_RATE,
        padding='longest',
        truncation=True,
        max_length=MAX_AUDIO_LENGTH,
        return_tensors='pt',
    )
    inputs['labels'] = examples['label']
    return inputs

print('Processing training data...')
if 'audio' in train_ds.column_names:
    train_ds = train_ds.map(
        preprocess_function,
        batched=True,
        batch_size=8,
        remove_columns=['audio'],
        desc='Preprocessing train'
    )
else:
    print('   train_ds already preprocessed, skipping.')

print('Processing validation data...')
if 'audio' in eval_ds.column_names:
    eval_ds = eval_ds.map(
        preprocess_function,
        batched=True,
        batch_size=8,
        remove_columns=['audio'],
        desc='Preprocessing eval'
    )
else:
    print('   eval_ds already preprocessed, skipping.')

print(f'\n✅ Preprocessing complete!')
print(f'   Training samples:   {len(train_ds)}')
print(f'   Validation samples: {len(eval_ds)}')
print(f'   Features: {train_ds.column_names}')

Processing training data...
   train_ds already preprocessed, skipping.
Processing validation data...
   eval_ds already preprocessed, skipping.

✅ Preprocessing complete!
   Training samples:   9729
   Validation samples: 2433
   Features: ['label', 'input_values', 'attention_mask', 'labels']


## Step 7: Set Up Training

Key settings for WavLM-large:
- `learning_rate=1e-5` — smaller than base model to protect pretrained weights
- `per_device_train_batch_size=4` — reduced for large model VRAM
- `gradient_accumulation_steps=8` — keeps effective batch size at 32
- `fp16=True` — essential for fitting large model in 16GB VRAM
- Early stopping with patience=3

In [31]:
import torch.nn as nn

accuracy_metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    refs        = eval_pred.label_ids
    acc = accuracy_metric.compute(predictions=predictions, references=refs)
    f1  = f1_score(refs, predictions, average='weighted')
    return {'accuracy': acc['accuracy'], 'f1': f1}

# Weighted loss to penalize TESS dominance
label_counts  = np.array([counts[i] for i in range(len(emotion_labels))], dtype=np.float32)
class_weights = torch.tensor(1.0 / label_counts)
class_weights = (class_weights / class_weights.sum() * len(emotion_labels)).to(device)

training_args = TrainingArguments(
    output_dir='./emotion-model',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=5e-6,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=15,
    warmup_ratio=0.05,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    fp16=True,                        
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    save_total_limit=2,
    report_to='none',
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits
        loss    = nn.CrossEntropyLoss(weight=class_weights)(logits, labels)
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=feature_extractor,
    compute_metrics=compute_metrics,
)

print('✅ Training setup complete!')
print('   Model:           WavLM-large (317M params)')
print('   Datasets:        RAVDESS + CREMA-D + TESS (capped) + SAVEE')
print('   Effective batch: 32 (4 × 8 accumulation steps)')
print('   Loss:            Weighted CrossEntropy (anti-TESS bias)')

✅ Training setup complete!
   Model:           WavLM-large (317M params)
   Datasets:        RAVDESS + CREMA-D + TESS + SAVEE
   Effective batch: 32 (4 × 8 accumulation steps)
   Continuing from: epoch 10 → epoch 15 (5 extra epochs)


## Step 8: Train the Model

Expected training time on Kaggle P100: **~3–4 hours**
Expected training time on Kaggle T4: **~4–5 hours**

Early stopping will stop automatically when validation accuracy stops improving.

In [ ]:
!pip install accelerate==0.34.0 --extra-index-url https://download.pytorch.org/whl/cu118 -q


In [ ]:
torch.backends.cudnn.enabled = False
print('Continuing training from epoch 10...')
print('=' * 60)

train_result = trainer.train(resume_from_checkpoint=True)  # ← add this

print('\n' + '=' * 60)
print('✅ Training complete!')
print(f'   Total training time: {train_result.metrics["train_runtime"]:.0f}s')

Continuing training from epoch 10...


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Epoch,Training Loss,Validation Loss


## Step 8b: Skip Training — Load a Previously Saved Model

Run this cell **instead of Step 8** to load a saved model.
Add your model zip as a Kaggle dataset input first.

In [30]:
import os

print("All files in /kaggle/input/:")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

All files in /kaggle/input/:
/kaggle/input/datasets/ziadmohamed11/emorec/config.json
/kaggle/input/datasets/ziadmohamed11/emorec/preprocessor_config.json
/kaggle/input/datasets/ziadmohamed11/emorec/label_mapping.json
/kaggle/input/datasets/ziadmohamed11/emorec/training_args.bin
/kaggle/input/datasets/ziadmohamed11/emorec/model.safetensors


In [31]:
import os
import shutil

# ✅ Correct source path matching your actual file locations
src_path = '/kaggle/input/datasets/ziadmohamed11/emorec'
dst_path = '/kaggle/working/emotion-recognition-final'

os.makedirs(dst_path, exist_ok=True)

for filename in os.listdir(src_path):
    shutil.copy2(
        os.path.join(src_path, filename),
        os.path.join(dst_path, filename)
    )

print(f'Copied files: {os.listdir(dst_path)}')

# Load from working directory
model_path        = dst_path
feature_extractor = AutoFeatureExtractor.from_pretrained(model_path)
model             = AutoModelForAudioClassification.from_pretrained(model_path)
model.eval()
model = model.to(device)

with open(f'{model_path}/label_mapping.json', 'r') as f:
    label_mapping = json.load(f)

id2label       = {int(k): v for k, v in label_mapping['id2label'].items()}
label2id       = label_mapping['label2id']
emotion_labels = id2label

print('✅ Model loaded!')
print('Emotions:', list(id2label.values()))

Copied files: ['config.json', 'preprocessor_config.json', 'model.safetensors', 'label_mapping.json', 'training_args.bin']


Loading weights:   0%|          | 0/492 [00:00<?, ?it/s]

✅ Model loaded!
Emotions: ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']


## Step 9: Evaluate the Model

In [20]:
print('Evaluating model...')
eval_results = trainer.evaluate()

print('\n' + '=' * 60)
print(f'  Final Validation Accuracy : {eval_results["eval_accuracy"]*100:.2f}%')
print(f'  Final Validation F1 Score : {eval_results["eval_f1"]*100:.2f}%')
print('=' * 60)

Evaluating model...


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(



  Final Validation Accuracy : 73.16%
  Final Validation F1 Score : 72.53%


## Step 10: Detailed Accuracy & Confusion Matrix

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns

print('Calculating detailed accuracy metrics...')
print('=' * 60)

predictions = trainer.predict(eval_ds)
preds       = np.argmax(predictions.predictions, axis=1)
labels      = predictions.label_ids

overall_accuracy = accuracy_metric.compute(predictions=preds, references=labels)
print(f'\nOVERALL ACCURACY: {overall_accuracy["accuracy"]*100:.2f}%')
print('=' * 60)

# Per-emotion accuracy
print('\nPER-EMOTION ACCURACY:')
print('-' * 60)
emotion_correct = {}
emotion_total   = {}

for pred, label in zip(preds, labels):
    emotion = id2label[label]
    if emotion not in emotion_total:
        emotion_total[emotion]   = 0
        emotion_correct[emotion] = 0
    emotion_total[emotion] += 1
    if pred == label:
        emotion_correct[emotion] += 1

for emotion in sorted(emotion_total.keys()):
    acc     = (emotion_correct[emotion] / emotion_total[emotion]) * 100
    correct = emotion_correct[emotion]
    total   = emotion_total[emotion]
    bar     = '█' * int(acc / 2.5) + '░' * (40 - int(acc / 2.5))
    print(f'{emotion:12s} [{bar}] {acc:5.1f}% ({correct}/{total})')

# Classification report
print('\nDETAILED CLASSIFICATION REPORT:')
print('-' * 60)
report = classification_report(
    labels, preds,
    target_names=[id2label[i] for i in range(len(id2label))],
    digits=3
)
print(report)

# Confusion matrix
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=[id2label[i] for i in range(len(id2label))],
    yticklabels=[id2label[i] for i in range(len(id2label))],
    cbar_kws={'label': 'Count'}
)
plt.xlabel('Predicted Emotion', fontsize=12, fontweight='bold')
plt.ylabel('True Emotion', fontsize=12, fontweight='bold')
plt.title('Confusion Matrix — Emotion Recognition Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nSUMMARY STATISTICS:')
print('=' * 60)
print(f'Total Test Samples:      {len(labels)}')
print(f'Correct Predictions:     {np.sum(preds == labels)}')
print(f'Incorrect Predictions:   {np.sum(preds != labels)}')
print(f'Overall Accuracy:        {overall_accuracy["accuracy"]*100:.2f}%')
print(f'Best Emotion:  {max(emotion_correct, key=lambda x: emotion_correct[x]/emotion_total[x]).upper()}')
print(f'Worst Emotion: {min(emotion_correct, key=lambda x: emotion_correct[x]/emotion_total[x]).upper()}')
print('=' * 60)

Calculating detailed accuracy metrics...


NameError: name 'trainer' is not defined

## Step 11: Save the Model

In [22]:
import shutil

# Save current best model
model_save_path = './emotion-recognition-final'
trainer.save_model(model_save_path)
feature_extractor.save_pretrained(model_save_path)

with open(f'{model_save_path}/label_mapping.json', 'w') as f:
    json.dump({'id2label': id2label, 'label2id': label2id}, f, indent=2)

# Zip directly into Kaggle output — no copy needed
shutil.make_archive(
    '/kaggle/working/emotion-recognition-final',  # destination
    'zip',
    '.',
    'emotion-recognition-final'
)

print('✅ Model saved!')
print('   Go to Output tab → download emotion-recognition-final.zip')

✅ Model saved!
   Go to Output tab → download emotion-recognition-final.zip


In [23]:
import os
import shutil

# First check what exists
print("Files in /kaggle/working/:")
for item in os.listdir('/kaggle/working/'):
    print(f"  {item}")

# Save model to the final folder
model_save_path = './emotion-recognition-final'
os.makedirs(model_save_path, exist_ok=True)

trainer.save_model(model_save_path)
feature_extractor.save_pretrained(model_save_path)

with open(f'{model_save_path}/label_mapping.json', 'w') as f:
    json.dump({'id2label': id2label, 'label2id': label2id}, f, indent=2)

print(f'\nFiles saved to {model_save_path}:')
for item in os.listdir(model_save_path):
    print(f'  {item}')

# Zip it
shutil.make_archive(
    '/kaggle/working/emotion-recognition-final',
    'zip',
    model_save_path
)

print('\n✅ Done!')
print(f'Zip size: {os.path.getsize("/kaggle/working/emotion-recognition-final.zip") / 1e9:.2f} GB')
print('Now look in Output tab — emotion-recognition-final.zip should appear')

Files in /kaggle/working/:
  emotion-recognition-final
  emotion-model
  .virtual_documents
  emotion-recognition-final.zip

Files saved to ./emotion-recognition-final:
  label_mapping.json
  training_args.bin
  config.json
  model.safetensors
  preprocessor_config.json

✅ Done!
Zip size: 1.17 GB
Now look in Output tab — emotion-recognition-final.zip should appear


In [29]:
from IPython.display import FileLink
FileLink('emotion-recognition-final.zip')

/kaggle/working/emotion-recognition-final.zip

In [25]:
# Run this after loading SAVEE to verify labels
print("SAVEE label distribution:")
savee_counts = Counter(savee["label"])
for label_id, count in sorted(savee_counts.items()):
    print(f"  {emotion_labels[label_id]:12s}: {count}")


SAVEE label distribution:
  neutral     : 120
  happy       : 60
  sad         : 60
  angry       : 60
  fearful     : 60
  disgust     : 60
  surprised   : 60


---
# 🔮 Prediction Section

## Step 12: Create Prediction Function

Model is loaded **once globally** — not reloaded on every prediction call.
This makes segment-by-segment analysis fast (~2 seconds per segment vs ~3 minutes).

In [5]:
model_path       = './emotion-recognition-final'
device           = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_AUDIO_LENGTH = 16000 * 5

print('Loading model for inference...')
_feature_extractor = AutoFeatureExtractor.from_pretrained(model_path)
_inference_model   = AutoModelForAudioClassification.from_pretrained(model_path)
_inference_model.eval()
_inference_model = _inference_model.to(device)

with open(f'{model_path}/label_mapping.json', 'r') as f:
    _label_mapping = json.load(f)

print('✅ Model loaded once — all predictions will reuse it!')


def predict_emotion(audio_path):
    audio, sr = librosa.load(audio_path, sr=_feature_extractor.sampling_rate)

    inputs = _feature_extractor(
        audio,
        sampling_rate=sr,
        return_tensors='pt',
        padding=True,
        max_length=MAX_AUDIO_LENGTH,
        truncation=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = _inference_model(**inputs).logits

    probabilities     = torch.nn.functional.softmax(logits, dim=-1)[0]
    predicted_id      = torch.argmax(probabilities).item()
    predicted_emotion = _label_mapping['id2label'][str(predicted_id)]
    confidence        = probabilities[predicted_id].item()

    emotion_scores = {
        _label_mapping['id2label'][str(idx)]: prob.item()
        for idx, prob in enumerate(probabilities)
    }

    return {
        'emotion'   : predicted_emotion,
        'confidence': confidence,
        'all_scores': emotion_scores
    }


def display_prediction(result):
    print('\n' + '=' * 60)
    print('EMOTION PREDICTION RESULTS')
    print('=' * 60)
    print(f'\nPredicted Emotion: {result["emotion"].upper()}')
    print(f'Confidence: {result["confidence"]*100:.2f}%')
    print('\n' + '-' * 60)
    print('All Emotion Scores:')
    print('-' * 60)
    for emotion, score in sorted(result['all_scores'].items(), key=lambda x: x[1], reverse=True):
        bar = '█' * int(score * 40) + '░' * (40 - int(score * 40))
        print(f'{emotion:12s} [{bar}] {score*100:5.2f}%')
    print('=' * 60 + '\n')


print('✅ Prediction functions ready!')

Loading model for inference...
✅ Model loaded once — all predictions will reuse it!
✅ Prediction functions ready!


## Step 13: Test on a Dataset Sample

In [27]:
import random

test_idx             = random.randint(0, len(combined) - 1)
original_test_sample = combined[test_idx]
true_emotion         = emotion_labels[original_test_sample['label']]

print(f'Test Sample #{test_idx}')
print(f'True Emotion: {true_emotion.upper()}')

audio_bytes  = original_test_sample['audio']['bytes']
audio_array, sampling_rate = sf.read(io.BytesIO(audio_bytes))
display(AudioPlayer(audio_array, rate=sampling_rate))

temp_path = '/tmp/test_audio.wav'
sf.write(temp_path, audio_array, sampling_rate)

result = predict_emotion(temp_path)
display_prediction(result)

if result['emotion'] == true_emotion:
    print('✅ CORRECT PREDICTION!')
else:
    print(f'❌ INCORRECT — Expected: {true_emotion.upper()}')

Test Sample #10476
True Emotion: NEUTRAL



EMOTION PREDICTION RESULTS

Predicted Emotion: NEUTRAL
Confidence: 83.66%

------------------------------------------------------------
All Emotion Scores:
------------------------------------------------------------
neutral      [█████████████████████████████████░░░░░░░] 83.66%
angry        [█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  3.47%
sad          [█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  3.22%
disgust      [█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  2.88%
calm         [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  2.44%
happy        [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  1.77%
fearful      [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  1.74%
surprised    [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  0.83%

✅ CORRECT PREDICTION!


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


## Step 14: Test on Your Own Audio File

On Kaggle: add your audio file as an input dataset, then update the path below.

In [16]:
from collections import Counter as AudioCounter

audio_file = '/kaggle/input/datasets/ziadmohamed11/arabic5/ht6ywils_QxiJtfmL.mp3'
print(f'Analysing: {audio_file}')

audio_data, sr = librosa.load(audio_file, sr=None)
display(AudioPlayer(audio_data, rate=sr))
duration = len(audio_data) / sr
print(f'\nAudio duration: {duration:.2f} seconds')

# ── Segment analysis ────────────────────────────────────────────────
print('\nAnalyzing emotions every 10 seconds...')
print('=' * 60)

segment_duration = 10
segment_samples  = int(segment_duration * sr)
total_segments   = int(np.ceil(len(audio_data) / segment_samples))

segment_emotions   = []
segment_confidence = []

for i in range(total_segments):
    start_sample = i * segment_samples
    end_sample   = min((i + 1) * segment_samples, len(audio_data))
    segment      = audio_data[start_sample:end_sample]

    temp_segment_path = f'/tmp/segment_{i}.wav'
    sf.write(temp_segment_path, segment, sr)

    result     = predict_emotion(temp_segment_path)
    start_time = i * segment_duration
    end_time   = min((i + 1) * segment_duration, duration)

    segment_emotions.append(result['emotion'])
    segment_confidence.append(result['confidence'])
    print(f'  {start_time:5.1f}s - {end_time:5.1f}s | Emotion: {result["emotion"].upper():12s} | Confidence: {result["confidence"]*100:5.2f}%')

print('=' * 60)

emotion_counts           = AudioCounter(segment_emotions)
major_emotion            = emotion_counts.most_common(1)[0][0]
major_emotion_count      = emotion_counts.most_common(1)[0][1]
major_emotion_percentage = (major_emotion_count / len(segment_emotions)) * 100

print('\nMAJOR EMOTION DETECTED:')
print('=' * 60)
print(f'   Emotion:   {major_emotion.upper()}')
print(f'   Frequency: {major_emotion_count}/{len(segment_emotions)} segments ({major_emotion_percentage:.1f}%)')
print('=' * 60)

print('\nEmotion Distribution:')
print('-' * 60)
for emotion, count in emotion_counts.most_common():
    percentage = (count / len(segment_emotions)) * 100
    bar        = '█' * int(percentage / 2.5) + '░' * (40 - int(percentage / 2.5))
    print(f'{emotion:12s} [{bar}] {percentage:5.1f}% ({count} segments)')
print('-' * 60)

print('\nOverall emotion prediction (entire audio):')
result = predict_emotion(audio_file)   # ✅ use original file directly
display_prediction(result)

y = audio_data

Analysing: /kaggle/input/datasets/ziadmohamed11/arabic5/ht6ywils_QxiJtfmL.mp3



Audio duration: 27.66 seconds

Analyzing emotions every 10 seconds...
    0.0s -  10.0s | Emotion: FEARFUL      | Confidence: 67.55%
   10.0s -  20.0s | Emotion: SAD          | Confidence: 56.52%
   20.0s -  27.7s | Emotion: FEARFUL      | Confidence: 68.66%

MAJOR EMOTION DETECTED:
   Emotion:   FEARFUL
   Frequency: 2/3 segments (66.7%)

Emotion Distribution:
------------------------------------------------------------
fearful      [██████████████████████████░░░░░░░░░░░░░░]  66.7% (2 segments)
sad          [█████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░]  33.3% (1 segments)
------------------------------------------------------------

Overall emotion prediction (entire audio):

EMOTION PREDICTION RESULTS

Predicted Emotion: FEARFUL
Confidence: 67.55%

------------------------------------------------------------
All Emotion Scores:
------------------------------------------------------------
fearful      [███████████████████████████░░░░░░░░░░░░░] 67.55%
sad          [████░░░░░░░░░░░░░░░░░░

## Step 15: Batch Prediction on Multiple Samples

In [ ]:
print('Testing on 10 random samples from the combined dataset...\n')

correct = 0
total   = 10

for i in range(total):
    idx          = random.randint(0, len(combined) - 1)
    sample       = combined[idx]
    true_emotion = emotion_labels[sample['label']]

    audio_bytes  = sample['audio']['bytes']
    audio_array, sampling_rate = sf.read(io.BytesIO(audio_bytes))

    temp_path = f'/tmp/test_{i}.wav'
    sf.write(temp_path, audio_array, sampling_rate)

    result     = predict_emotion(temp_path)
    is_correct = result['emotion'] == true_emotion
    if is_correct:
        correct += 1

    status = '✅' if is_correct else '❌'
    print(f'{status} Sample {i+1:2d}: True={true_emotion:10s} | Predicted={result["emotion"]:10s} | Confidence={result["confidence"]*100:5.2f}%')

accuracy = (correct / total) * 100
print(f'\nAccuracy on {total} samples: {accuracy:.1f}% ({correct}/{total})')

## Step 16: Dataset Bias Detection

Tests model accuracy separately on each dataset to check if it's biased towards CREMA-D.

In [30]:
from collections import defaultdict

print('Testing bias towards specific datasets...')
print('=' * 60)

def evaluate_on_dataset(dataset, dataset_name, num_samples=100):
    indices  = np.random.choice(len(dataset), min(num_samples, len(dataset)), replace=False)
    correct  = 0
    total    = 0
    per_emotion_correct = defaultdict(int)
    per_emotion_total   = defaultdict(int)

    for idx in indices:
        sample       = dataset[int(idx)]
        audio_bytes  = sample['audio']['bytes']
        true_label   = sample['label']
        true_emotion = emotion_labels[true_label]

        audio_array, s = sf.read(io.BytesIO(audio_bytes))
        sf.write('/tmp/test_bias.wav', audio_array, s)

        result = predict_emotion('/tmp/test_bias.wav')
        per_emotion_total[true_emotion] += 1
        if result['emotion'] == true_emotion:
            correct += 1
            per_emotion_correct[true_emotion] += 1
        total += 1

    accuracy = correct / total * 100
    print(f'\n{dataset_name}')
    print(f'   Overall Accuracy: {accuracy:.1f}% ({correct}/{total})')
    for emotion in sorted(per_emotion_total.keys()):
        c   = per_emotion_correct[emotion]
        t   = per_emotion_total[emotion]
        a   = c / t * 100 if t > 0 else 0
        bar = '█' * int(a / 2.5) + '░' * (40 - int(a / 2.5))
        print(f'     {emotion:12s} [{bar}] {a:5.1f}% ({c}/{t})')
    return accuracy

# Reload each dataset for bias testing
ravdess_bias = load_dataset('xbgoose/ravdess').cast_column('audio', Audio(decode=False))['train'].map(label_ravdess).select_columns(['audio', 'label']).cast_column('label', Value('int64'))
cremad_bias  = load_dataset('mteb/crema-d', split='train').cast_column('audio', Audio(decode=False)).cast_column('label', Value('int64')).map(label_cremad).select_columns(['audio', 'label'])
tess_bias    = load_dataset('AbstractTTS/TESS', split='train').cast_column('audio', Audio(decode=False)).map(label_tess).cast_column('label', Value('int64')).filter(lambda x: x['label'] != -1).select_columns(['audio', 'label'])
savee_bias   = load_dataset('AbstractTTS/SAVEE', split='train').cast_column('audio', Audio(decode=False)).map(label_savee).cast_column('label', Value('int64')).filter(lambda x: x['label'] != -1).select_columns(['audio', 'label'])

print('\n' + '=' * 60)
print('DATASET BIAS DETECTION RESULTS')
print('=' * 60)

ravdess_acc = evaluate_on_dataset(ravdess_bias, 'RAVDESS',  num_samples=100)
cremad_acc  = evaluate_on_dataset(cremad_bias,  'CREMA-D',  num_samples=100)
tess_acc    = evaluate_on_dataset(tess_bias,    'TESS',     num_samples=100)
savee_acc   = evaluate_on_dataset(savee_bias,   'SAVEE',    num_samples=100)

print('\n' + '=' * 60)
print('SUMMARY:')
print('=' * 60)
datasets = {'RAVDESS': ravdess_acc, 'CREMA-D': cremad_acc, 'TESS': tess_acc, 'SAVEE': savee_acc}
for name, acc in sorted(datasets.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(acc / 2.5) + '░' * (40 - int(acc / 2.5))
    print(f'  {name:10s} [{bar}] {acc:.1f}%')

gap   = max(datasets.values()) - min(datasets.values())
best  = max(datasets, key=datasets.get)
worst = min(datasets, key=datasets.get)
print(f'\n  Best:  {best} ({datasets[best]:.1f}%)')
print(f'  Worst: {worst} ({datasets[worst]:.1f}%)')
print(f'  Gap:   {gap:.1f}%')

if gap > 15:
    print(f'\n  🔴 STRONG BIAS towards {best} — consider capping or weighted loss')
elif gap > 8:
    print(f'\n  🟡 MILD BIAS towards {best} — monitor but may not need fixing')
else:
    print(f'\n  🟢 NO SIGNIFICANT BIAS — model generalizes well')

Testing bias towards specific datasets...


Casting the dataset:   0%|          | 0/1440 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2800 [00:00<?, ? examples/s]

Filter:   0%|          | 0/480 [00:00<?, ? examples/s]


DATASET BIAS DETECTION RESULTS


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(



RAVDESS
   Overall Accuracy: 63.0% (63/100)
     angry        [█████████████████████████████░░░░░░░░░░░]  72.7% (8/11)
     calm         [███████████████████████████████████░░░░░]  87.5% (7/8)
     disgust      [█████████████████████░░░░░░░░░░░░░░░░░░░]  54.2% (13/24)
     fearful      [██████████████████████████░░░░░░░░░░░░░░]  66.7% (8/12)
     happy        [████████████████████████░░░░░░░░░░░░░░░░]  61.1% (11/18)
     neutral      [█████████████████████████████████░░░░░░░]  83.3% (5/6)
     sad          [██████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]  16.7% (2/12)
     surprised    [████████████████████████████████████████] 100.0% (9/9)

CREMA-D
   Overall Accuracy: 66.0% (66/100)
     angry        [█████████████████████████████████████░░░]  92.9% (13/14)
     disgust      [███████████████████████░░░░░░░░░░░░░░░░░]  57.9% (11/19)
     fearful      [██████████████████░░░░░░░░░░░░░░░░░░░░░░]  45.5% (10/22)
     happy        [█████████████████████████░░░░░░░░░░░░░░░]  63.2% (12/19)
     ne